In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("DeltaLakeAssignment").getOrCreate()
print("Spark Session Created Successfully !!")

Spark Session Created Successfully !!


Cell 2
Create Database

In [0]:
data = [
    (1, "John", "Sales", 50000),
    (2, "Alice", "HR", 45000),
    (3, "Bob", "IT", None),
    (4, "David", "Finance",60000),
    (5, None, "Marketing", 48000)
]

columns = ["EmpID", "Name", "Department","Salary"]
df = spark.createDataFrame(data, columns)
display(df)

EmpID,Name,Department,Salary
1,John,Sales,50000
2,Alice,HR,45000
3,Bob,IT,null
4,David,Finance,60000
5,null,Marketing,48000


Cell 3
Data Cleaning


In [0]:
clean_df = df.na.drop()
clean_df = clean_df.dropDuplicates()
display(clean_df)

EmpID,Name,Department,Salary
1,John,Sales,50000
2,Alice,HR,45000
4,David,Finance,60000


Cell 4
Save Data as Delta Table

In [0]:
clean_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("employee_delta")

Cell 5 Display Delta Table

In [0]:
display(spark.table("employee_delta"))

EmpID,Name,Department,Salary
1,John,Sales,50000
2,Alice,HR,45000
4,David,Finance,60000


Cell 6 Import Delta Library

In [0]:
from delta.tables import DeltaTable

deltaTable = DeltaTable.forName(spark, "employee_delta")

Cell 7 Create Incremental Dataset

In [0]:
new_data = [
    (2, "Alice", "HR", 48000),
    (3, "BOB", "IT", 52000),
    (6, "Emma", "Sales",55000),
    (7,"Chris","finance", 61000)
]
new_df = spark.createDataFrame(new_data, columns)
display(new_df)

EmpID,Name,Department,Salary
2,Alice,HR,48000
3,BOB,IT,52000
6,Emma,Sales,55000
7,Chris,finance,61000


Cell 8 Merge Operation

In [0]:
(
    deltaTable.alias("old")
    .merge(
        new_df.alias("new"),
        "old.EmpID = new.EmpID"
    )
    .whenMatchedUpdate(set={
        "Name": "new.Name",
        "Department":"new.Department",
        "Salary": "new.Salary",
    })
    .whenNotMatchedInsert(values={
        "EmpID": "new.EmpID",
        "Name" :"new.Name",
        "Department": "new.Department",
        "Salary": "new.Salary"
    })
    .execute()
)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

Cell 9: Display Final Dataset

In [0]:
display(spark.table("employee_delta"))

EmpID,Name,Department,Salary
1,John,Sales,50000
4,David,Finance,60000
2,Alice,HR,48000
3,BOB,IT,52000
6,Emma,Sales,55000
7,Chris,finance,61000


Cell 10: Validate Results

In [0]:
from pyspark.sql.functions import col

final_df = spark.table("employee_delta")

print("Total Rows:", final_df.count())

duplicates = (
    final_df.groupBy("EmpID")
    .count()
    .filter(col("count") > 1)
)

display(duplicates)

Total Rows: 6


EmpID,count


Cell 11: Summary

In [0]:
print("================================")
print("Delta Lake Incremental Processing")
print("Assignment Completed Successfully")
print("================================")

print("Total Rows:", final_df.count())

final_df.show(truncate=False)

Delta Lake Incremental Processing
Assignment Completed Successfully
Total Rows: 6
+-----+-----+----------+------+
|EmpID|Name |Department|Salary|
+-----+-----+----------+------+
|1    |John |Sales     |50000 |
|4    |David|Finance   |60000 |
|2    |Alice|HR        |48000 |
|3    |BOB  |IT        |52000 |
|6    |Emma |Sales     |55000 |
|7    |Chris|finance   |61000 |
+-----+-----+----------+------+

